In [1]:
import os
import glob
import json
import itertools
import numpy as np
import pandas as pd
from pathlib import Path
from PNW_cmap import PNW_cmap
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from vip_slap2_analysis.utils.utils import save_figure
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.glutamate.summary import GlutamateSummary
from vip_slap2_analysis.utils.utils import normalize
from vip_slap2_analysis.glutamate.analysis import (
    GlutamateAnalysisConfig,
    run_glutamate_analysis,
    resolve_glutamate_analysis_paths
)

import seaborn as sns
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))



In [2]:
%load_ext autoreload
%autoreload 2

%matplotlib notebook

In [3]:
savepath = r'C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Data_Club\April2026\figures'

In [4]:
target_mice = [
    803496,
    804730,804733,810196,
    809047,803121,
    826033,838410,834788
]

registry = VIPSessionRegistry.from_basepath(
    r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
)

process_df = registry.sessions(
    subject_ids=target_mice,
    exclude_session_types=["expression_check", "volume_imaging"],
    paradigms=["change_detection_passive"],
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

print(f"Loaded {len(assets)} session assets")

Loaded 56 session assets


In [5]:
st_paths = [glob.glob(os.path.join(asset.derived_dir,'**','glutamate_single_trial_df.npz'),recursive=True)[0] for asset in assets]

In [6]:
act_summary_path = r"C:\Users\andrew.shelton\Downloads\activation_summary.csv"
act_summary = pd.read_csv(act_summary_path)

In [11]:
config = GlutamateAnalysisConfig(
    tuning_method="fve",                  # or "fve" / "manova"
    tuning_fve_mode="trace",                # new
    tuning_fve_amplitude_func="mean",       # not very important for trace mode, harmless
    tuning_fve_sample_slice=(50, 100),      # used for time_avg mode; safe to leave here
    tuning_response_classes=("activated",),
    manova_stat="Wilks' lambda",
    n_shuffles_tuning=100,
    manova_max_timepoints=10,
    manova_use_post_only=True,
    random_seed=0,
)

In [ ]:
all_tuning_summary = []
all_tuning_per_image = []

for asset in assets:
    print(asset.session_id)
    session_root = asset.session_dir
    
    results = run_glutamate_analysis(session_root, config=config)
    
    tuning_summary = results["tuning_summary_table"].copy()
    tuning_per_image = results["tuning_per_image_table"].copy()
    
    # add session-level depth metadata
    dmd_depth_map = {
        "DMD1": asset.metadata.get("dmd1_depth", np.nan),
        "DMD2": asset.metadata.get("dmd2_depth", np.nan),
    }
    
    tuning_summary["depth"] = tuning_summary["dmd"].map(dmd_depth_map)
    tuning_per_image["depth"] = tuning_per_image["dmd"].map(dmd_depth_map)
    
    # optional: keep asset/session info
    tuning_summary["session_name"] = getattr(asset, "session_name", os.path.basename(session_root))
    tuning_per_image["session_name"] = getattr(asset, "session_name", os.path.basename(session_root))
    
    all_tuning_summary.append(tuning_summary)
    all_tuning_per_image.append(tuning_per_image)

tuning_summary_table = pd.concat(all_tuning_summary, ignore_index=True)
tuning_per_image_table = pd.concat(all_tuning_per_image, ignore_index=True)

803496_2025-07-25_13-02-10
803496_2025-07-28_08-04-39


C:\Users\andrew.shelton\Dropbox\allen institute\Python_Code\ams\ophys\vip-slap2-analysis\src\vip_slap2_analysis\glutamate\analysis.py:384: RuntimeWarning: Mean of empty slice
  return np.nanmean(arr, axis=1)
C:\Users\andrew.shelton\Dropbox\allen institute\Python_Code\ams\ophys\vip-slap2-analysis\src\vip_slap2_analysis\glutamate\analysis.py:384: RuntimeWarning: Mean of empty slice
  return np.nanmean(arr, axis=1)


803496_2025-07-29_13-34-35
803496_2025-07-30_10-05-23
803496_2025-07-31_09-43-28
